# ASM2D-TSN Simulation

This notebook runs the ASM2D-TSN simulation model from the source package and persists the generated dataset and metadata under `data/asm2d-tsn/simulation`.

The reported output variables here are exclusively the configured composites.

In [ ]:
import pandas as pd
from src.models.simulation.asm2d_tsn_simulation import (
    run_asm2d_tsn_simulation,
    sweep_asm2d_tsn_operating_space,
 )

simulation_result = run_asm2d_tsn_simulation(
    save_artifacts=True,
    include_debug_data=True,
    show_progress=True,
    progress_description="ASM2D-TSN dataset",
)

dataset = simulation_result["dataset"]
metadata = simulation_result["metadata"]
artifact_paths = simulation_result["artifact_paths"]
petersen_matrix = simulation_result["petersen_matrix"]
composition_matrix = simulation_result["composition_matrix"]
effluent_states = simulation_result["effluent_states"]
solver_diagnostics = simulation_result["solver_diagnostics"]
solver_summary = simulation_result["solver_summary"]

print(f"Generated {len(dataset)} rows for {metadata['simulation_name']}.")
print(f"Dataset saved to: {artifact_paths['dataset_csv']}")
print(f"Metadata saved to: {artifact_paths['metadata_json']}")
print(f"Petersen matrix shape: {petersen_matrix.shape}")
print(f"Composition matrix shape: {composition_matrix.shape}")
print("Saved dataset contract remains composite-only; debug payloads stay in memory.")

output_columns = [column_name for column_name in metadata["dependent_columns"] if column_name.startswith("Out_")]
print(f"Reported output variables: {', '.join(output_columns)}")

display(dataset.head())
display(pd.Series(metadata, name="value").to_frame())
display(pd.json_normalize(solver_summary, sep="."))
display(solver_diagnostics.head())
display(effluent_states.head())
display(pd.DataFrame(petersen_matrix, index=metadata["processes"], columns=metadata["state_columns"]))
display(pd.DataFrame(composition_matrix, index=metadata["measured_output_columns"], columns=metadata["state_columns"]))

ASM2D-TSN dataset:   0%|          | 0/10000 [00:00<?, ?sample/s]

In [ ]:
import numpy as np
import pandas as pd
from scipy.linalg import null_space

from src.utils.process import has_active_projection

icsor_constraint_basis = null_space(petersen_matrix)
icsor_A_matrix = icsor_constraint_basis.T

icsor_A_matrix = np.round(icsor_A_matrix, 5)
icsor_A_matrix[np.abs(icsor_A_matrix) < 1e-10] = 0.0

for row_index in range(icsor_A_matrix.shape[0]):
    non_zero_entries = icsor_A_matrix[row_index, icsor_A_matrix[row_index, :] != 0]
    if len(non_zero_entries) > 0:
        icsor_A_matrix[row_index, :] = icsor_A_matrix[row_index, :] / non_zero_entries[0]

macroscopic_stoichiometric_matrix = petersen_matrix @ composition_matrix.T
measured_constraint_basis = null_space(macroscopic_stoichiometric_matrix)
A_matrix = measured_constraint_basis.T

A_matrix = np.round(A_matrix, 5)
A_matrix[np.abs(A_matrix) < 1e-10] = 0.0

for row_index in range(A_matrix.shape[0]):
    non_zero_entries = A_matrix[row_index, A_matrix[row_index, :] != 0]
    if len(non_zero_entries) > 0:
        A_matrix[row_index, :] = A_matrix[row_index, :] / non_zero_entries[0]

classical_projection_active = has_active_projection(A_matrix)
measured_output_dimension = int(A_matrix.shape[1])
measured_nullity = int(A_matrix.shape[0])
macroscopic_rank = int(np.linalg.matrix_rank(macroscopic_stoichiometric_matrix))
classical_projection_status = pd.DataFrame(
    [
        {
            "projection_active": classical_projection_active,
            "constraint_status": "active" if classical_projection_active else "inactive_trivial_null_space",
            "measured_output_dimension": measured_output_dimension,
            "rank_S_macro": macroscopic_rank,
            "nullity_S_macro": measured_nullity,
        }
    ]
)

print(f"Fractional Petersen matrix shape: {petersen_matrix.shape}")
print(f"icsor invariant matrix shape: {icsor_A_matrix.shape}")
print(f"Measured-space invariant matrix shape kept for downstream regressors: {A_matrix.shape}")
if classical_projection_active:
    print("Classical measured-space projection remains active because the measured-output null space is non-trivial.")
else:
    print("Classical measured-space projection is inactive because null_space(nu I_comp^T) is trivial in the measured-output basis.")
    print("Projected classical results and measured-space mass-balance discrepancy tables are suppressed downstream.")

display(
    pd.DataFrame(
        icsor_A_matrix,
        index=[f"constraint_{index + 1}" for index in range(icsor_A_matrix.shape[0])],
        columns=metadata["state_columns"],
    )
)
display(
    pd.DataFrame(
        A_matrix,
        index=[f"constraint_{index + 1}" for index in range(A_matrix.shape[0])],
        columns=metadata["measured_output_columns"],
    )
)
display(classical_projection_status)

Fractional Petersen matrix shape: (28, 21)
icsor invariant matrix shape: (6, 21)
Measured-space invariant matrix shape kept for downstream regressors: (0, 6)


,S_A,S_F,S_I,S_N2,S_NH4,S_NO2,S_NO3,S_PO4,S_ALK,S_O2,...,X_S,X_H,X_PAO,X_PP,X_PHA,X_AOB,X_NOB,X_TSS,X_MeOH,X_MeP
constraint_1,-0.0,1.0,-122.115741,43.285494,43.180556,43.391975,43.391975,-29.893519,1.476852,-0.0,...,6.891975,8.584877,8.584877,-6.334877,4.367284,8.584877,8.584877,-7.279321,2.435185,-2.280864
constraint_2,0.0,1.0,41.259080,30.538741,30.934625,30.142857,30.142857,8.578692,-5.548426,0.0,...,-1.389831,0.099274,0.099274,-3.207022,-2.156174,0.099274,0.099274,3.593220,77.556901,55.619855
constraint_3,0.0,1.0,-17.963673,7.606458,9.781029,5.431887,5.431887,78.202825,-30.441978,0.0,...,-17.095863,-15.973764,-15.973764,-1.039354,-14.537841,-15.973764,-15.973764,24.229062,-13.890010,-1.048436
constraint_4,0.0,1.0,50.768730,34.358306,33.546145,35.171553,35.171553,-3.445168,11.373507,0.0,...,1.364821,2.915309,2.915309,-2.984799,0.017372,2.915309,2.915309,-0.029316,-52.864278,-38.073833
constraint_5,-0.0,1.0,0.197026,7.572491,11.695167,3.449814,3.449814,79.107807,-57.717472,-0.0,...,56.650558,57.769517,57.769517,316.583643,44.460967,57.769517,57.769517,-74.100372,-20.063197,23.252788
constraint_6,-0.0,1.0,-31.996269,-6.250000,-30.932836,18.432836,18.432836,107.679104,345.544776,-0.0,...,-13.630597,-12.731343,-12.731343,56.078358,-11.656716,-12.731343,-12.731343,19.425373,11.552239,26.917910


,COD,TN,TKN,TP,TSS,VSS


## Solver Sweep

This section samples the configured operating space using Latin Hypercube Sampling (LHS) without changing the saved dataset contract. It summarizes how often the current acceptance threshold is met, how often the multistart guess wins, and how often dynamic relaxation is needed.

In [2]:
SWEEP_SAMPLE_COUNT = 1024

sweep_result = sweep_asm2d_tsn_operating_space(
    n_samples=SWEEP_SAMPLE_COUNT,
    random_seed=int(metadata["random_seed"]),
    show_progress=True,
    progress_description="ASM2D-TSN operating-space sweep",
)

sweep_summary = sweep_result["summary"]
sweep_diagnostics = sweep_result["solver_diagnostics"].copy()
sweep_diagnostics["threshold_margin"] = (
    sweep_diagnostics["acceptance_threshold"] - sweep_diagnostics["residual_max"]
)

print(f"Operating-space sweep completed for {sweep_summary['sample_count']} sampled points.")
print(
    f"Accepted points at threshold {sweep_summary['acceptance_threshold']}: "
    f"{sweep_summary['accepted_count']} ({sweep_summary['accepted_rate']:.1%})"
 )
print(f"Multistart selected rate: {sweep_summary['multistart_selected_rate']:.1%}")
print(f"Dynamic relaxation used rate: {sweep_summary['dynamic_relaxation_used_rate']:.1%}")
print(f"Dynamic relaxation improved rate: {sweep_summary['dynamic_relaxation_improved_rate']:.1%}")

display(pd.json_normalize(sweep_summary, sep="."))
display(sweep_diagnostics.sort_values("residual_max", ascending=False).head(20))

ASM2D-TSN operating-space sweep: 100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆ| 1024/1024 [02:34<00:00,  6.61sample/s]

Operating-space sweep completed for 1024 sampled points.
Accepted points at threshold 5.0: 1023 (99.9%)
Multistart selected rate: 41.2%
Dynamic relaxation used rate: 23.9%
Dynamic relaxation improved rate: 23.9%


,sample_count,accepted_count,accepted_rate,acceptance_threshold,multistart_selected_rate,dynamic_relaxation_used_rate,dynamic_relaxation_improved_rate,mean_attempt_count,max_attempt_count,residual_max_quantiles.q50,...,residual_max_quantiles.q95,residual_max_quantiles.q99,residual_max_quantiles.max,nfev_quantiles.q50,nfev_quantiles.q90,nfev_quantiles.q95,nfev_quantiles.max,selected_strategy_counts.multistart,selected_strategy_counts.initial,selected_strategy_counts.dynamic_relaxation
0,1024,1023,0.999023,5.0,0.412109,0.239258,0.239258,1.0,1,7.460699e-14,...,4.472867e-13,3.707676,4.902331,55.0,184.7,264.8,1200.0,422,357,245


,success,accepted,status,nfev,residual_l2,residual_max,acceptance_threshold,initial_residual_max,multistart_residual_max,selected_strategy,dynamic_relaxation_used,dynamic_relaxation_improved,sample_index,HRT,Aeration,threshold_margin
855,True,True,2,52,5.423108,4.902331,5.0,4.902331,85.122371,initial,False,False,855,20.261283,1.347782,0.097669
56,True,True,2,27,4.679744,4.679744,5.0,23.210990,4.679744,multistart,False,False,56,34.365597,1.284969,0.320256
865,True,True,2,48,5.648156,4.485051,5.0,27.279447,4.485051,multistart,False,False,865,7.085130,2.358315,0.514949
322,True,True,3,83,6.621399,4.308687,5.0,9.356358,4.308687,multistart,False,False,322,10.845658,1.698756,0.691313
739,True,True,3,88,5.811170,4.289413,5.0,4.505957,4.289413,multistart,False,False,739,25.959236,1.216410,0.710587
582,True,True,3,279,6.330710,4.128436,5.0,4.128436,66.285225,initial,False,False,582,24.201750,0.732427,0.871564
819,True,True,3,148,6.322393,4.006370,5.0,9.530505,4.006370,multistart,False,False,819,13.200848,2.107519,0.993630
134,True,True,3,64,3.996509,3.955088,5.0,4.251298,3.955088,multistart,False,False,134,28.825165,1.781891,1.044912
207,True,True,2,50,3.822045,3.822045,5.0,3.822045,37.953454,initial,False,False,207,33.489061,1.670620,1.177955
308,True,True,2,50,6.603762,3.751906,5.0,3.751906,182.064660,initial,False,False,308,7.343085,1.782836,1.248094
